# Image2GPS Baseline Data and Dataloaders

This notebook loads our train/validation data from the Hugging Face Dataset repo and sets up the baseline-style PyTorch dataset and dataloaders. It then trains and evaluates a new model on the data (needs GPU access) and saves the outputs.

In [ ]:
# Uncomment and run this cell if your kernel is missing dependencies.
# %pip install datasets huggingface_hub torch torchvision numpy pillow matplotlib


# Data

## Loading the Train and Validation Datasets

Here we now load our uploaded Hugging Face Dataset directly instead of reading from the local `data/image2gps_dataset/` folder.


In [ ]:
HF_DATASET_REPO_ID = "mhedlund/CIS5190IMG2GPS"
print("Dataset repo:", HF_DATASET_REPO_ID)


In [ ]:
from datasets import load_dataset, Image

dataset_train = load_dataset(HF_DATASET_REPO_ID, split="train")
dataset_validation = load_dataset(HF_DATASET_REPO_ID, split="validation")


In [ ]:
print(dataset_train)
print(dataset_validation)


In [ ]:
dataset_train[0]

## Defining the Custom Dataset Class

This is the same `GPSImageDataset` pattern from the baseline notebook.

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np


class GPSImageDataset(Dataset):
    def __init__(self, hf_dataset, transform=None, lat_mean=None, lat_std=None, lon_mean=None, lon_std=None):
        self.hf_dataset = hf_dataset
        self.transform = transform

        # Compute mean and std from the dataframe if not provided
        self.latitude_mean = lat_mean if lat_mean is not None else np.mean(np.array(self.hf_dataset['Latitude']))
        self.latitude_std = lat_std if lat_std is not None else np.std(np.array(self.hf_dataset['Latitude']))
        self.longitude_mean = lon_mean if lon_mean is not None else np.mean(np.array(self.hf_dataset['Longitude']))
        self.longitude_std = lon_std if lon_std is not None else np.std(np.array(self.hf_dataset['Longitude']))

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        # Extract data
        example = self.hf_dataset[idx]

        # Load and process the image
        image = example['image'].convert('RGB')
        latitude = example['Latitude']
        longitude = example['Longitude']
        # image = image.rotate(-90, expand=True)
        if self.transform:
            image = self.transform(image)

        # Normalize GPS coordinates
        latitude = (latitude - self.latitude_mean) / self.latitude_std
        longitude = (longitude - self.longitude_mean) / self.longitude_std
        gps_coords = torch.tensor([latitude, longitude], dtype=torch.float32)

        return image, gps_coords

In [ ]:
from torchvision.transforms import InterpolationMode

In [ ]:
transform = transforms.Compose([
    transforms.RandomResizedCrop(224),  # Random crop and resize to 224x224
    transforms.RandomHorizontalFlip(),  # Random horizontal flip
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Random color jitter
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Optionally, you can create a separate transform for inference without augmentations
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
# Create the training dataset and dataloader
train_dataset = GPSImageDataset(hf_dataset=dataset_train, transform=transform)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Retrieve normalization parameters from the training dataset
lat_mean = train_dataset.latitude_mean
lat_std = train_dataset.latitude_std
lon_mean = train_dataset.longitude_mean
lon_std = train_dataset.longitude_std

print("lat_mean:", lat_mean)
print("lat_std:", lat_std)
print("lon_mean:", lon_mean)
print("lon_std:", lon_std)

In [ ]:
# Create the validation dataset and dataloader using training mean and std
val_dataset = GPSImageDataset(
    hf_dataset=dataset_validation,
    transform=inference_transform,
    lat_mean=lat_mean,
    lat_std=lat_std,
    lon_mean=lon_mean,
    lon_std=lon_std
)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)


## Batch Shape Check

Confirms the image tensors and normalized GPS labels are the expected format.

In [ ]:
images, gps_coords = next(iter(train_dataloader))

print("image batch shape:", images.shape)
print("gps batch shape:", gps_coords.shape)
print("first normalized gps label:", gps_coords[0])

In [ ]:
# Convert normalized labels back to raw latitude/longitude for inspection.
gps_coords_denorm = gps_coords * torch.tensor([lat_std, lon_std]) + torch.tensor([lat_mean, lon_mean])
gps_coords_denorm[:5]

## Visualize Data

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
import numpy as np


def denormalize(tensor, mean, std):
    mean = np.array(mean)
    std = np.array(std)
    tensor = tensor.numpy().transpose((1, 2, 0))  # Convert from C x H x W to H x W x C
    tensor = std * tensor + mean
    tensor = np.clip(tensor, 0, 1)
    return tensor


data_iter = iter(train_dataloader)
images, gps_coords = next(data_iter)
gps_coords_real = gps_coords * torch.tensor([lat_std, lon_std]) + torch.tensor([lat_mean, lon_mean])

num_examples = min(8, len(images))
for i in range(num_examples):
    image = denormalize(images[i], mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    plt.imshow(image)
    plt.title(
        f"Latitude: {gps_coords_real[i, 0].item():.6f}, "
        f"Longitude: {gps_coords_real[i, 1].item():.6f}"
    )
    plt.axis("off")
    plt.show()


## Next Step

At this point, `train_dataloader` and `val_dataloader` are ready to use for actual model code. Feel free to adjust the transforms or other parts of the pipeline if you think it will make the models behave better in the end!


## Model Architecture

We use ResNet18 as a regression model with 2 outputs.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

In [ ]:
class ResNet18LocationRegressor(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = models.resnet18(
            weights=models.ResNet18_Weights.IMAGENET1K_V1
        )

        in_features = self.model.fc.in_features

        self.model.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 2)   # output: normalized [latitude, longitude]
        )

        self.register_buffer("lat_mean", torch.tensor(0.0))
        self.register_buffer("lat_std", torch.tensor(1.0))
        self.register_buffer("lon_mean", torch.tensor(0.0))
        self.register_buffer("lon_std", torch.tensor(1.0))

    def forward(self, x):
        return self.model(x)

In [ ]:
# use average haversine distance for loss

class HaversineLoss(nn.Module):
    def __init__(self, lat_mean, lat_std, lon_mean, lon_std):
        super().__init__()

        self.lat_mean = lat_mean
        self.lat_std = lat_std
        self.lon_mean = lon_mean
        self.lon_std = lon_std

    def forward(self, preds_norm, targets_norm):
        # Denormalize
        pred_lat = preds_norm[:, 0] * self.lat_std + self.lat_mean
        pred_lon = preds_norm[:, 1] * self.lon_std + self.lon_mean

        true_lat = targets_norm[:, 0] * self.lat_std + self.lat_mean
        true_lon = targets_norm[:, 1] * self.lon_std + self.lon_mean

        # Convert degrees to radians
        pred_lat = torch.deg2rad(pred_lat)
        pred_lon = torch.deg2rad(pred_lon)
        true_lat = torch.deg2rad(true_lat)
        true_lon = torch.deg2rad(true_lon)

        # Haversine formula
        dlat = pred_lat - true_lat
        dlon = pred_lon - true_lon

        a = (
            torch.sin(dlat / 2) ** 2
            + torch.cos(true_lat) * torch.cos(pred_lat) * torch.sin(dlon / 2) ** 2
        )

        earth_radius_m = 6371000

        # Keep the asin/sqrt inputs numerically valid.
        a = torch.clamp(a, 0.0, 1.0)

        distance = 2 * earth_radius_m * torch.arcsin(torch.sqrt(a))

        return distance.mean()

In [ ]:
# to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet18LocationRegressor().to(device)

criterion = HaversineLoss(lat_mean, lat_std, lon_mean, lon_std)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

## Training and Validation Loop

In [ ]:
num_epochs = 10
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------
    # Training
    # ------------------
    model.train()
    train_loss = 0.0

    for images, gps_coords in train_dataloader:
        images = images.to(device)
        gps_coords = gps_coords.to(device)

        optimizer.zero_grad()

        preds = model(images)
        loss = criterion(preds, gps_coords)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_dataloader.dataset)

    # ------------------
    # Validation
    # ------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for images, gps_coords in val_dataloader:
            images = images.to(device)
            gps_coords = gps_coords.to(device)

            preds = model(images)
            loss = criterion(preds, gps_coords)

            val_loss += loss.item() * images.size(0)

    val_loss /= len(val_dataloader.dataset)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        print(f"--> New best model found. Haversine Distance: {best_val_loss:.4f} meters. Saving to model.pt")

        model.lat_mean.fill_(lat_mean)
        model.lat_std.fill_(lat_std)
        model.lon_mean.fill_(lon_mean)
        model.lon_std.fill_(lon_std)
        
        torch.save(model.state_dict(), 'model.pt')
        
    print("------------------------------")

    

## View Outputs

In [ ]:
model = ResNet18LocationRegressor().to(device)

#Load the healthy weights from the disk
state_dict = torch.load('model.pt', map_location='cpu')
model.load_state_dict(state_dict)
model.to(device)

print("Success: model has been reset to the best saved weights.")

model.eval()

num_examples = 5

images, gps_coords = next(iter(val_dataloader))

images = images[:num_examples].to(device)
gps_coords = gps_coords[:num_examples]

with torch.no_grad():
    preds_norm = model(images).cpu()

stats_mean = torch.tensor([lat_mean, lon_mean])
stats_std = torch.tensor([lat_std, lon_std])

preds_real = preds_norm * stats_std + stats_mean
gps_real = gps_coords * stats_std + stats_mean

for i in range(num_examples):
    img = images[i].cpu()
    img = denormalize(img, mean=[0.485, 0.456, 0.406], 
                      std=[0.229, 0.224, 0.225])

    plt.imshow(img)
    plt.title(f"Pred: {preds_real[i].tolist()}\nTrue: {gps_real[i].tolist()}")
    plt.axis("off")
    plt.show()

## Save model.pt

In [ ]:
from pathlib import Path

model_path = Path("model.pt")
print("model.pt path:", model_path.resolve())
print("model.pt exists:", model_path.exists())


In [ ]:
# Optional Colab-only copy to Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# !cp model.pt /content/drive/MyDrive/model.pt


In [ ]:
import torch
try:
    # Load with map_location="cpu" to ensure it works regardless of GPU availability
    state_dict = torch.load('model.pt', map_location='cpu')
    print("Success: model.pt loaded.")
    
    # Print the first few keys to make sure they look like layer names
    print("Sample keys:", list(state_dict.keys())[:5])
except Exception as e:
    print(f"Error: Could not load model.pt. Details: {e}")

## Professor-Released Validation Set

The professors provided a 100-image Hugging Face dataset, `gydou/released_img`, that should be closer to the hidden leaderboard test distribution than our internal validation split. We run this section after training so it can evaluate the saved `model.pt` checkpoint on that released validation set.

In [ ]:
RELEASED_DATASET_REPO_ID = "gydou/released_img"

released_dataset = load_dataset(RELEASED_DATASET_REPO_ID, split="train")
print(released_dataset)
print("columns:", released_dataset.column_names)
released_dataset[0]

In [ ]:
def standardize_gps_columns(ds):
    lat_aliases = ["Latitude", "latitude", "lat"]
    lon_aliases = ["Longitude", "longitude", "lon", "lng"]

    lat_col = next((col for col in lat_aliases if col in ds.column_names), None)
    lon_col = next((col for col in lon_aliases if col in ds.column_names), None)

    if lat_col is None or lon_col is None:
        raise ValueError(f"Could not find GPS label columns in {ds.column_names}")

    if lat_col != "Latitude":
        ds = ds.rename_column(lat_col, "Latitude")
    if lon_col != "Longitude":
        ds = ds.rename_column(lon_col, "Longitude")
    return ds


released_dataset = standardize_gps_columns(released_dataset)

released_val_dataset = GPSImageDataset(
    hf_dataset=released_dataset,
    transform=inference_transform,
    lat_mean=lat_mean,
    lat_std=lat_std,
    lon_mean=lon_mean,
    lon_std=lon_std,
)
released_val_dataloader = DataLoader(released_val_dataset, batch_size=32, shuffle=False)

print("released validation examples:", len(released_val_dataset))

In [ ]:
from pathlib import Path

# Use the best checkpoint saved during training if it exists.
if Path("model.pt").exists():
    state_dict = torch.load("model.pt", map_location="cpu")
    model.load_state_dict(state_dict)
    model.to(device)
    print("Loaded model.pt for professor-released validation evaluation.")
else:
    print("model.pt not found; evaluating the current in-memory model instead.")

model.eval()

In [ ]:
def haversine_distance_m(preds_norm, targets_norm, lat_mean, lat_std, lon_mean, lon_std):
    pred_lat = preds_norm[:, 0] * lat_std + lat_mean
    pred_lon = preds_norm[:, 1] * lon_std + lon_mean
    true_lat = targets_norm[:, 0] * lat_std + lat_mean
    true_lon = targets_norm[:, 1] * lon_std + lon_mean

    pred_lat = torch.deg2rad(pred_lat)
    pred_lon = torch.deg2rad(pred_lon)
    true_lat = torch.deg2rad(true_lat)
    true_lon = torch.deg2rad(true_lon)

    dlat = pred_lat - true_lat
    dlon = pred_lon - true_lon
    a = (
        torch.sin(dlat / 2) ** 2
        + torch.cos(true_lat) * torch.cos(pred_lat) * torch.sin(dlon / 2) ** 2
    )
    a = torch.clamp(a, 0.0, 1.0)
    return 2 * 6_371_000 * torch.arcsin(torch.sqrt(a))


def evaluate_gps_model(model, dataloader):
    all_preds_norm = []
    all_targets_norm = []

    model.eval()
    with torch.no_grad():
        for images, gps_coords in dataloader:
            images = images.to(device)
            preds_norm = model(images).cpu()
            all_preds_norm.append(preds_norm)
            all_targets_norm.append(gps_coords.cpu())

    preds_norm = torch.cat(all_preds_norm)
    targets_norm = torch.cat(all_targets_norm)

    model_errors_m = haversine_distance_m(
        preds_norm, targets_norm, lat_mean, lat_std, lon_mean, lon_std
    )
    train_mean_preds_norm = torch.zeros_like(targets_norm)
    train_mean_errors_m = haversine_distance_m(
        train_mean_preds_norm, targets_norm, lat_mean, lat_std, lon_mean, lon_std
    )

    stats_mean = torch.tensor([lat_mean, lon_mean])
    stats_std = torch.tensor([lat_std, lon_std])
    preds_real = preds_norm * stats_std + stats_mean
    targets_real = targets_norm * stats_std + stats_mean

    return preds_real, targets_real, model_errors_m, train_mean_errors_m


released_preds, released_targets, released_errors_m, released_mean_errors_m = evaluate_gps_model(
    model, released_val_dataloader
)

print(f"Released validation model mean error: {released_errors_m.mean().item():.2f} meters")
print(f"Released validation model median error: {released_errors_m.median().item():.2f} meters")
print(f"Train-mean GPS baseline mean error: {released_mean_errors_m.mean().item():.2f} meters")
print(f"Train-mean GPS baseline median error: {released_mean_errors_m.median().item():.2f} meters")

In [ ]:
import pandas as pd

released_results_df = pd.DataFrame(
    {
        "pred_lat": released_preds[:, 0].numpy(),
        "pred_lon": released_preds[:, 1].numpy(),
        "true_lat": released_targets[:, 0].numpy(),
        "true_lon": released_targets[:, 1].numpy(),
        "error_m": released_errors_m.numpy(),
        "train_mean_baseline_error_m": released_mean_errors_m.numpy(),
    }
)

display(released_results_df.describe())
display(released_results_df.sort_values("error_m", ascending=False).head(10))